In [1]:
# Kaggle already has torch, numpy. Just install SUMO-specific stuff.
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    print(result.stdout)

# Install SUMO via PPA
run("sudo add-apt-repository -y ppa:sumo/stable")
run("sudo apt-get update -qq")
run("sudo apt-get install -y sumo sumo-tools -qq")

# Python packages
run("pip install -q traci sumolib libsumo 'tensorboard==2.20.0' 'setuptools==69.5.1'")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,497 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,998 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-dr

In [5]:
import os

os.environ["SUMO_HOME"] = "/usr/share/sumo"
os.environ["PYTHONPATH"] = f"/usr/share/sumo/tools:{os.environ.get('PYTHONPATH', '')}"

# Critical: redirect ALL output here
OUTPUT_DIR = "/kaggle/output"
os.makedirs(f"{OUTPUT_DIR}/results", exist_ok=True)

In [6]:
import shutil, os
from kaggle_secrets import UserSecretsClient

# Remove existing repo if present
if os.path.exists("/kaggle/working/repo"):
    shutil.rmtree("/kaggle/working/repo")

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

BRANCH = "mixer-fix"

run(f"git clone -b {BRANCH} https://{token}@github.com/Akiri017/Mono-QMIX.git /kaggle/working/repo")

os.chdir("/kaggle/working/repo")

In [7]:
import os

# Check what's actually in /kaggle/working
print(os.listdir("/kaggle/working"))

['repo', 'results', '.virtual_documents']


In [8]:
import os, shutil

# Clean up existing results if present from previous session
if os.path.exists("/kaggle/working/results"):
    shutil.rmtree("/kaggle/working/results")

os.makedirs("/kaggle/working/results", exist_ok=True)

# Remove existing results in repo and replace with symlink to /kaggle/working/results
if os.path.exists("/kaggle/working/repo/results") or os.path.islink("/kaggle/working/repo/results"):
    shutil.rmtree("/kaggle/working/repo/results", ignore_errors=True)

os.symlink("/kaggle/working/results", "/kaggle/working/repo/results")
print("Symlink created → /kaggle/working/results")

Symlink created → /kaggle/working/results


In [9]:
import os
print(os.path.islink("/kaggle/working/repo/results"))
print(os.readlink("/kaggle/working/repo/results"))

True
/kaggle/working/results


In [10]:
import os

assert os.path.islink("/kaggle/working/repo/results"), "Symlink not created!"
assert os.readlink("/kaggle/working/repo/results") == "/kaggle/working/results", "Wrong target!"

with open("/kaggle/working/repo/results/test.txt", "w") as f:
    f.write("ok")

assert os.path.exists("/kaggle/working/results/test.txt"), "Symlink not working!"
os.remove("/kaggle/working/results/test.txt")

print("Symlink verified. Safe to run training.")

Symlink verified. Safe to run training.


In [11]:
import os, subprocess, shutil

os.chdir("/kaggle/working/repo")

# Re-create symlink if lost
if not os.path.islink("/kaggle/working/repo/results"):
    os.makedirs("/kaggle/working/results", exist_ok=True)
    if os.path.exists("/kaggle/working/repo/results"):
        shutil.rmtree("/kaggle/working/repo/results")
    os.symlink("/kaggle/working/results", "/kaggle/working/repo/results")
    print("Symlink re-created.")
else:
    print("Symlink intact.")

process = subprocess.Popen(
    "python run_experiments.py --seeds 5 --t_max 500000 --eval_episodes 20",
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd="/kaggle/working/repo"
)

for line in process.stdout:
    print(line, end="", flush=True)

process.wait()
print(f"\nExit code: {process.returncode}")

Symlink intact.
2026-04-10 10:18:30.780885: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775816310.979251    2182 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775816311.045421    2182 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775816311.541728    2182 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775816311.541775    2182 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775816311.541779    2182 computation_placer.cc:177] comput

In [16]:
import os
for root, dirs, files in os.walk("/kaggle/working/results"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/working/results/eval/summary_20260410_164507.json
/kaggle/working/results/models/seed5/step_300000/reward_stats.pth
/kaggle/working/results/models/seed5/step_300000/mixer.pth
/kaggle/working/results/models/seed5/step_300000/optimizer.pth
/kaggle/working/results/models/seed5/step_300000/agent.pth
/kaggle/working/results/models/seed5/step_300000/training_state.json
/kaggle/working/results/models/seed5/step_200000/reward_stats.pth
/kaggle/working/results/models/seed5/step_200000/mixer.pth
/kaggle/working/results/models/seed5/step_200000/optimizer.pth
/kaggle/working/results/models/seed5/step_200000/agent.pth
/kaggle/working/results/models/seed5/step_200000/training_state.json
/kaggle/working/results/models/seed5/step_100000/reward_stats.pth
/kaggle/working/results/models/seed5/step_100000/mixer.pth
/kaggle/working/results/models/seed5/step_100000/optimizer.pth
/kaggle/working/results/models/seed5/step_100000/agent.pth
/kaggle/working/results/models/seed5/step_100000/training_state

In [15]:
run("git status")

On branch mixer-fix
Your branch is up to date with 'origin/mixer-fix'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	pymarl/src/results/eval/greedy_shortest_seed5.json
	pymarl/src/results/eval/noop_seed5.json
	pymarl/src/results/eval/qmix_seed5.json
	pymarl/src/results/eval/random_seed5.json
	pymarl/src/results/logs/events.out.tfevents.1775816326.e340280dccf7.2182.0

nothing added to commit but untracked files present (use "git add" to track)



In [18]:
import shutil, os
from kaggle_secrets import UserSecretsClient

os.chdir("/kaggle/working/repo")

# Configure git
run("git config --global user.email 'mariannesantos174@gmail.com'")
run("git config --global user.name 'Akiri017'")

# Copy results into repo directly
shutil.copytree("/kaggle/working/results", "/kaggle/working/repo/results_data", dirs_exist_ok=True)

run("git add results_data/")
run(f"git commit -m 'Add smoke test results seed5 500k'")

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
run(f"git remote set-url origin https://{token}@github.com/Akiri017/Mono-QMIX.git")
run(f"git push origin {BRANCH}")

print(f"Results pushed to branch: {BRANCH}")




[mixer-fix e2bc473] Add smoke test results seed5 500k
 59 files changed, 259 insertions(+)
 create mode 100644 results_data/eval/summary_20260410_164507.json
 create mode 100644 results_data/models/seed5/best/agent.pth
 create mode 100644 results_data/models/seed5/best/mixer.pth
 create mode 100644 results_data/models/seed5/best/optimizer.pth
 create mode 100644 results_data/models/seed5/best/reward_stats.pth
 create mode 100644 results_data/models/seed5/final/agent.pth
 create mode 100644 results_data/models/seed5/final/mixer.pth
 create mode 100644 results_data/models/seed5/final/optimizer.pth
 create mode 100644 results_data/models/seed5/final/reward_stats.pth
 create mode 100644 results_data/models/seed5/step_100000/agent.pth
 create mode 100644 results_data/models/seed5/step_100000/mixer.pth
 create mode 100644 results_data/models/seed5/step_100000/optimizer.pth
 create mode 100644 results_data/models/seed5/step_100000/reward_stats.pth
 create mode 100644 results_data/models/se

In [19]:
import os
from kaggle_secrets import UserSecretsClient

os.chdir("/kaggle/working/repo")

run("git add pymarl/src/results/")
run("git commit -m 'Add pymarl eval and logs seed5 500k'")

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
run(f"git remote set-url origin https://{token}@github.com/Akiri017/Mono-QMIX.git")
run(f"git push origin {BRANCH}")

print(f"pymarl results pushed to branch: {BRANCH}")


[mixer-fix 5a98854] Add pymarl eval and logs seed5 500k
 5 files changed, 960 insertions(+)
 create mode 100644 pymarl/src/results/eval/greedy_shortest_seed5.json
 create mode 100644 pymarl/src/results/eval/noop_seed5.json
 create mode 100644 pymarl/src/results/eval/qmix_seed5.json
 create mode 100644 pymarl/src/results/eval/random_seed5.json
 create mode 100644 pymarl/src/results/logs/events.out.tfevents.1775816326.e340280dccf7.2182.0



pymarl results pushed to branch: mixer-fix
